<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/FT2_Fine_tuning_with_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Fine-tuning an LLM
Fine-tuning refers to the process of taking a pre-trained Large Language Model (LLM) and training it further on a smaller, specialized dataset to adapt it for a specific task or domain. By exposing them to domain-specific data, we can enhance their ability to tackle specialized tasks—whether it’s for classification, writing legal contracts, or processing medical diagnoses with greater accuracy.


##Full Fine-tuning
Full fine-tuning involves taking a pre-trained language model and updating all of its trainable parameters (weights and biases) on your new, task-specific dataset. The entire model architecture remains the same; only the values of the weights are adjusted during the training process to better perform on the downstream task.

While full fine-tuning is a straightforward approach, it can be computationally expensive (especially for large models) and may lead to overfitting on small datasets.


##Parameter-Efficient Fine-Tuning (PEFT)
These methods aim to achieve performance comparable to full fine-tuning while only training a small fraction of the model's parameters. This significantly reduces computational cost and storage requirements. Some popular PEFT techniques include:

* Low-Rank Adaptation (LoRA): Introduces small, low-rank matrices alongside the original weights. Only these low-rank matrices are trained, while the original pre-trained weights are kept frozen.


* QLoRA (Quantization-aware Low-Rank Adaptation): Combines quantization (reducing the precision of weights) with LoRA to further reduce memory footprint during training.

* Instruction Tuning: This is a specific type of fine-tuning where the model is trained on a dataset of tasks described in natural language instructions. The goal is to make the model better at following instructions and generalizing to new tasks it hasn't seen explicitly during fine-tuning. This often involves formatting the training data as (instruction, input, output) triplets.

* Reinforcement Learning from Human Feedback (RLHF): While not strictly a fine-tuning method in the traditional supervised sense, RLHF is a crucial technique for aligning large language models with human preferences. It typically involves several stages, including supervised fine-tuning followed by training a reward model based on human comparisons of different model outputs, and finally using reinforcement learning to optimize the language model to maximize this reward.


#Full Fine-tuning
We will first conduct full fine-tuning and then explore PEFT.

##Install Necessary Libraries
* transformers: For working with pre-trained models and datasets.
* datasets: For easily creating and managing small datasets.

In [ ]:
!pip install transformers datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict # Import DatasetDict here
import random
import torch

##Load the Dataset:
We will load the Financial Phrasebank dataset and then take a small random sample.

In [ ]:
ds = load_dataset("warwickai/financial_phrasebank_mirror")

After loading the dataset, it's good practice to inspect its structure and content. We'll convert the dataset to a Pandas DataFrame for easier viewing and display the first few rows to understand the data format, which includes 'sentence' and 'label' columns.

In [ ]:
import pandas as pd

# Convert the 'train' split of the dataset to a pandas DataFrame
df = ds["train"].to_pandas()

# Display the first 5 rows of the DataFrame
display(df.head())

For the purpose of this example, we will work with a smaller subset of the data. We will randomly sample 400 records from the training split to speed up the fine-tuning process. This helps in demonstrating the concepts without requiring extensive computational resources.

In [ ]:
# Take a random sample of 400 records from the 'train' split of the dataset
# First, shuffle the dataset, then select the first 400 records
ds_train = ds["train"].shuffle(seed=42).select(range(400))

print(ds_train)
print(f"Number of records in the random sample: {len(ds_train)}")

To verify the sampling and observe the structure of the dataset after sampling, we will display the first 5 entries of the `ds_train` dataset. This allows us to see the 'sentence' and 'label' fields for these records.

In [ ]:
from IPython.display import Markdown

In [ ]:
display(Markdown(str(ds_train[:5])))

#Load a Tiny Pre-trained Language Model and Tokenizer

In [ ]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

##Architecture of the Base Model

In [ ]:
# Print layer names
for name, param in model.named_parameters():
    print(name)

##Number of Parameters in the Base Model

In [ ]:
# Calculate the total number of parameters
total_params = sum(p.numel() for p in model.parameters())

# Print the result
print(f"Total number of parameters: {total_params}")

#Preprocess the Dataset:
Now, we need to format our data for the language model. We format our data as:

sentence: `[sentence]` label: `[label]`



In [ ]:
def preprocess_function(examples):
    inputs = [f"sentence: {sentence} label: {label}{tokenizer.eos_token}" for sentence, label in zip(examples['sentence'], examples['label'])]
    # Tokenize inputs and add 'labels' key
    tokenized_inputs = tokenizer(inputs, truncation=True, padding='max_length', max_length=128)
    # Shift labels to align with model's expected input format (Causal LM)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

tokenized_dataset = ds_train.map(preprocess_function, batched=True)

##View Prepared Data

In [ ]:
for i in range(5):  # Print the first 5 examples
       example = tokenized_dataset[i]
       decoded_text = tokenizer.decode(example['input_ids'])
       print(f"Example {i + 1}:")
       print(decoded_text)
       print(example)  # Print the full dictionary for the example
       print("-" * 20)

#Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_emotion_model",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_dir="./logs",
    report_to="none",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    save_strategy="epoch"
)

#Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=lambda data: tokenizer.pad(data, padding='max_length', max_length=128, return_tensors='pt')
    # Call the pad method with appropriate arguments within a lambda function
)

#Fine-tune the Model

In [ ]:
trainer.train()

Once the full fine-tuning process is complete, it's essential to save the trained model and its tokenizer. This allows us to reuse the fine-tuned model later without having to retrain it, and ensures consistency in tokenization when loading the model.

In [ ]:
# Define a path to save the fully fine-tuned model
save_path_full_ft = "./fully_fine_tuned_model_saved"

# Save the current state of the 'model' object
model.save_pretrained(save_path_full_ft)
tokenizer.save_pretrained(save_path_full_ft)

print(f"Fully fine-tuned model saved to: {save_path_full_ft}")

#Test the Fine-tuned Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Define the path where the fully fine-tuned model was saved
save_path_full_ft = "./fully_fine_tuned_model_saved"

# Load the fully fine-tuned model and tokenizer
loaded_fine_tuned_model = AutoModelForCausalLM.from_pretrained(save_path_full_ft)
loaded_fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path_full_ft)

# Move the loaded model to the appropriate device (e.g., GPU if available)
loaded_fine_tuned_model.to(model.device)

# Quick test for the fully fine-tuned model
user_input = input("Enter a sentence to predict its label: ")

# Format the input sentence into the prompt expected by the model
prompt = f"sentence: {user_input} label:"

# Tokenize the prompt using the loaded tokenizer
encoded_prompt = loaded_fine_tuned_tokenizer(prompt, return_tensors="pt")
input_ids = encoded_prompt["input_ids"].to(loaded_fine_tuned_model.device)
attention_mask = encoded_prompt["attention_mask"].to(loaded_fine_tuned_model.device)

# Generate a prediction using the loaded model
output = loaded_fine_tuned_model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=5, # Expecting a short label (e.g., a single digit)
    num_return_sequences=1,
    pad_token_id=loaded_fine_tuned_tokenizer.eos_token_id
)
generated_text = loaded_fine_tuned_tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True)

# Extract the predicted label from the generated text
predicted_label = "N/A"
try:
    # Assuming the model generates a simple digit as the label
    predicted_label = generated_text.strip().split()[0]
    # Optionally convert to int if always numeric
    # predicted_label = int(predicted_label)
except (ValueError, IndexError):
    pass # Keep as "N/A" if parsing fails

print(f"Input Sentence: {user_input}")
print(f"Predicted Label: {predicted_label}")
print("-------------------------------------")

Now, we will evaluate the performance of our fully fine-tuned model on the entire sampled dataset. We will iterate through each example, generate a prediction, and then calculate metrics such as accuracy and a detailed classification report using `sklearn.metrics`. This will give us a comprehensive understanding of how well the model performs on our specific task.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Create a list to store predictions and actual labels
full_ft_predictions = []
full_ft_actual_labels = []

# Iterate through the tokenized_dataset to get predictions from the full_fine_tuned_model
for i in range(len(tokenized_dataset)):
    sample_entry = tokenized_dataset[i]
    sample_sentence = sample_entry['sentence']
    actual_label = sample_entry['label']

    prompt = f"sentence: {sample_sentence} label:"
    encoded_prompt = tokenizer(prompt, return_tensors="pt")
    input_ids = encoded_prompt["input_ids"].to(loaded_fine_tuned_model.device)
    attention_mask = encoded_prompt["attention_mask"].to(loaded_fine_tuned_model.device)

    output = loaded_fine_tuned_model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=5,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_text = tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True)

    # Extract predicted label and handle potential errors
    try:
        predicted_label = generated_text.strip().split()[0]
        predicted_label = int(predicted_label)
    except (ValueError, IndexError):
        predicted_label = -1 # Sentinel for malformed predictions

    full_ft_predictions.append(predicted_label)
    full_ft_actual_labels.append(actual_label)

# Filter out invalid predictions for metric calculation
valid_indices_full_ft = [i for i, pred in enumerate(full_ft_predictions) if pred in [0, 1, 2]]
filtered_full_ft_predictions = [full_ft_predictions[i] for i in valid_indices_full_ft]
filtered_full_ft_actual_labels = [full_ft_actual_labels[i] for i in valid_indices_full_ft]

# Calculate accuracy and print report
if filtered_full_ft_actual_labels:
    accuracy_full_ft = accuracy_score(filtered_full_ft_actual_labels, filtered_full_ft_predictions)
    print(f"\nFull Fine-tuned Model Accuracy: {accuracy_full_ft:.4f}")

    report_full_ft = classification_report(
        filtered_full_ft_actual_labels,
        filtered_full_ft_predictions,
        target_names=['negative', 'neutral', 'positive'],
        zero_division=0
    )
    print("\nFull Fine-tuned Model Classification Report:")
    print(report_full_ft)
else:
    print("No valid predictions to calculate metrics for the Full Fine-tuned Model.")

# Adding predictions as a new column 'FT_label' to the tokenized_dataset
if 'FT_label' not in tokenized_dataset.column_names:
    tokenized_dataset = tokenized_dataset.add_column("FT_label", full_ft_predictions)
else:
    print("Column 'FT_label' already exists. Updating it.")
    tokenized_dataset = tokenized_dataset.remove_columns("FT_label")
    tokenized_dataset = tokenized_dataset.add_column("FT_label", full_ft_predictions)


In [ ]:
# Display the first few entries to show 'label' and 'FT_label' side-by-side
print("\nFirst 5 entries with original 'label' and new 'FT_label':")
for i in range(5):
    entry = tokenized_dataset[i]
    print(f"Sentence: {entry['sentence']}")
    print(f"Original Label: {entry['label']}")
    print(f"Predicted FT_label: {entry['FT_label']}")
    print("--------------------")

#Parameter-Efficient Fine-Tuning (PEFT)
We will now explore PEFT. We will continue to use the same dataset and the same LLM.


To implement Parameter-Efficient Fine-Tuning (PEFT) using LoRA, we need to import `LoraConfig` and `get_peft_model` from the `peft` library. `LoraConfig` defines the parameters for our LoRA setup, and `get_peft_model` applies these configurations to our base model.

In [ ]:
!pip install accelerate peft

In [ ]:
from peft import LoraConfig, get_peft_model
import torch

**Steps including loading and preparing the data, loading the LLM and Tokenizer remain the same as before.**

##Define Training Arguments

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM", # Specify the task type
    target_modules=["attn.c_attn", "attn.c_proj"], # Adjust based on the model architecture
)

# Wrap the base model with the LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # See how many parameters are trainable

##Create the Trainer

Similar to full fine-tuning, we initialize the `Trainer` class for LoRA. This `Trainer` will orchestrate the training loop, including managing epochs, batches, and logging, using the LoRA-wrapped model and the specified `TrainingArguments`.

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_fine_tuned_emotion_model",
    per_device_train_batch_size=4,
    num_train_epochs=50, # Increased epochs for stronger learning
    logging_dir="./logs",
    report_to="none",
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    save_strategy="epoch"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=lambda data: tokenizer.pad(data, padding='max_length', max_length=128, return_tensors='pt')
    # Call the pad method with appropriate arguments within a lambda function
)

##Train the Model

In [ ]:
trainer.train()

After the LoRA fine-tuning is complete, we save the LoRA adapter weights and the tokenizer. This allows us to load the base model and then attach the LoRA adapters to it for inference or further training, which is much more efficient than saving the entire model.

In [ ]:
# Define a path to save the fully fine-tuned model
save_path_full_ft = "./lora_fine_tuned_model_saved"

# Save the current state of the 'model' object
model.save_pretrained(save_path_full_ft)
tokenizer.save_pretrained(save_path_full_ft)

print(f"Fully fine-tuned model saved to: {save_path_full_ft}")

To test the LoRA fine-tuned model, we first need to load the original base model and then load the LoRA adapter weights on top of it. This process effectively reconstructs the fine-tuned model. We'll then use it to predict the label for a user-provided sentence, similar to how we tested the fully fine-tuned model.

To get a clear picture of the model's performance on individual examples, we'll display the original sentence, its true label, and the predicted label from our fully fine-tuned model for a few 5 entries of the dataset. This helps in qualitatively assessing the model's predictions.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel # Import PeftModel

# Define the path where the LoRA fine-tuned model was saved
save_path_lora_ft = "./lora_fine_tuned_model_saved"

# 1. Load the original base model
# Assuming 'model_name' (e.g., "distilgpt2") is still defined from earlier cells
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# 2. Load the PEFT adapter onto the base model
loaded_fine_tuned_model = PeftModel.from_pretrained(base_model, save_path_lora_ft)

# 3. Load the tokenizer (which was saved with the PEFT model)
loaded_fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path_lora_ft)

# Move the loaded model to the appropriate device (e.g., GPU if available)
loaded_fine_tuned_model.to(model.device)

# Quick test for the LoRA fine-tuned model
user_input = input("Enter a sentence to predict its label: ")

# Format the input sentence into the prompt expected by the model
prompt = f"sentence: {user_input} label:"

# Tokenize the prompt using the loaded tokenizer
encoded_prompt = loaded_fine_tuned_tokenizer(prompt, return_tensors="pt")
input_ids = encoded_prompt["input_ids"].to(loaded_fine_tuned_model.device)
attention_mask = encoded_prompt["attention_mask"].to(loaded_fine_tuned_model.device)

# Generate a prediction using the loaded model
output = loaded_fine_tuned_model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=5, # Expecting a short label (e.g., a single digit)
    num_return_sequences=1,
    pad_token_id=loaded_fine_tuned_tokenizer.eos_token_id
)
generated_text = loaded_fine_tuned_tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True)

# Extract the predicted label from the generated text
predicted_label = "N/A"
try:
    # Assuming the model generates a simple digit as the label
    predicted_label = generated_text.strip().split()[0]
    # Optionally convert to int if always numeric
    # predicted_label = int(predicted_label)
except (ValueError, IndexError):
    pass # Keep as "N/A" if parsing fails

print(f"Input Sentence: {user_input}")
print(f"Predicted Label: {predicted_label}")
print("-------------------------------------")

Finally, we evaluate the performance of the LoRA fine-tuned model on the entire sampled dataset. We'll iterate through each example, generate a prediction, and then calculate metrics such as accuracy and a detailed classification report. This allows us to compare its performance against the fully fine-tuned model and assess the effectiveness of PEFT.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel # Import PeftModel

# Define the path where the LoRA fine-tuned model was saved
save_path_lora_ft = "./lora_fine_tuned_model_saved"

# 1. Load the original base model
# Assuming 'model_name' (e.g., "distilgpt2") is still defined from earlier cells
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# 2. Load the PEFT adapter onto the base model
loaded_lora_fine_tuned_model = PeftModel.from_pretrained(base_model, save_path_lora_ft)

# 3. Load the tokenizer (which was saved with the PEFT model)
loaded_lora_fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path_lora_ft)

# Move the loaded model to the appropriate device (e.g., GPU if available)
loaded_lora_fine_tuned_model.to(model.device)

# Create a list to store predictions and actual labels
lora_ft_predictions = []
lora_ft_actual_labels = []

# Iterate through the tokenized_dataset to get predictions from the LoRA fine-tuned model
for i in range(len(tokenized_dataset)):
    sample_entry = tokenized_dataset[i]
    sample_sentence = sample_entry['sentence']
    actual_label = sample_entry['label']

    prompt = f"sentence: {sample_sentence} label:"
    encoded_prompt = loaded_lora_fine_tuned_tokenizer(prompt, return_tensors="pt")
    input_ids = encoded_prompt["input_ids"].to(loaded_lora_fine_tuned_model.device)
    attention_mask = encoded_prompt["attention_mask"].to(loaded_lora_fine_tuned_model.device)

    output = loaded_lora_fine_tuned_model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=2, # Reduced max_new_tokens, expecting just the digit and maybe a space
        num_return_sequences=1,
        do_sample=False, # Force greedy decoding for deterministic output
        pad_token_id=loaded_lora_fine_tuned_tokenizer.eos_token_id
    )
    generated_text = loaded_lora_fine_tuned_tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True)

    # Debugging: Print the raw generated text for each sample
    print(f"Sample: {i}, Sentence: '{sample_sentence}', Actual: {actual_label}, Raw Generated: '{generated_text}'")

    # Extract predicted label and handle potential errors
    try:
        predicted_label = generated_text.strip().split()[0]
        predicted_label = int(predicted_label)
    except (ValueError, IndexError):
        predicted_label = -1 # Sentinel for malformed predictions

    lora_ft_predictions.append(predicted_label)
    lora_ft_actual_labels.append(actual_label)

# Filter out invalid predictions for metric calculation
valid_indices_lora_ft = [i for i, pred in enumerate(lora_ft_predictions) if pred in [0, 1, 2]]
filtered_lora_ft_predictions = [lora_ft_predictions[i] for i in valid_indices_lora_ft]
filtered_lora_ft_actual_labels = [lora_ft_actual_labels[i] for i in valid_indices_lora_ft]

# Calculate accuracy and print report
if filtered_lora_ft_actual_labels:
    accuracy_lora_ft = accuracy_score(filtered_lora_ft_actual_labels, filtered_lora_ft_predictions)
    print(f"\nLoRA Fine-tuned Model Accuracy: {accuracy_lora_ft:.4f}")

    report_lora_ft = classification_report(
        filtered_lora_ft_actual_labels,
        filtered_lora_ft_predictions,
        target_names=['negative', 'neutral', 'positive'],
        zero_division=0
    )
    print("\nLoRA Fine-tuned Model Classification Report:")
    print(report_lora_ft)
else:
    print("No valid predictions to calculate metrics for the LoRA Fine-tuned Model.")

# Check if the 'LoRA_FT_label' column already exists before adding it
if 'LoRA_FT_label' not in tokenized_dataset.column_names:
    tokenized_dataset = tokenized_dataset.add_column("LoRA_FT_label", lora_ft_predictions)
else:
    print("Column 'LoRA_FT_label' already exists. Updating it.")
    tokenized_dataset = tokenized_dataset.remove_columns("LoRA_FT_label")
    tokenized_dataset = tokenized_dataset.add_column("LoRA_FT_label", lora_ft_predictions)

print("\nFirst 5 entries with original 'label' and new 'LoRA_FT_label':")
for i in range(5):
    entry = tokenized_dataset[i]
    print(f"Sentence: {entry['sentence']}")
    print(f"Original Label: {entry['label']}")
    print(f"Predicted LoRA_FT_label: {entry['LoRA_FT_label']}")
    print("--------------------")

With PEFT, the performance is very poor. In this case, it is because of the base model we chose.

**distilgpt2 (Generative Pre-trained Transformer 2):**

**Architecture**: It's a decoder-only Transformer model. This means it processes text sequentially, predicting the next word based on all the previous words. Its strength is in text generation and completing sequences.
**Training Objective:** Its primary goal during pre-training was Causal Language Modeling (predicting the next token).
Why it's not ideal for **Classification**: When you try to use it for classification (like outputting a single number), you're essentially forcing it to do text generation where the 'next word' must be your label. As you've seen, it struggles to constrain its output to just a single digit and often continues generating more text, because its core function is fluency, not discrete categorization.

In the next notebook we will try **distilbert-base-uncased (Distilled Bidirectional Encoder Representations from Transformers):**

**Architecture:** It's an encoder-only Transformer model. It processes the entire input sentence at once, understanding the context of each word based on all other words in the sentence (bidirectionally).
**Training Objective:** Its primary goals were Masked Language Modeling (predicting masked words) and Next Sentence Prediction. This trains it to build rich, contextual representations of sentences.
Why it's ideal for Classification: For classification, a small classification head (a few linear layers) is added on top of DistilBERT's encoder. This head takes the model's contextual understanding of the entire sentence and maps it directly to a fixed number of output classes (e.g., 0, 1, 2 for sentiment). It's designed to understand the input and output a category, not to generate new text.